## Classical ML Model #3: **Gradient Boosting Classifier**

First, we will import our pre-split and scaled training, validation, and test sets.

In [184]:
import pandas as pd
import joblib

# Load the preprocessed and scaled data
X_train_scaled = joblib.load('processed_data/X_train_scaled.pkl')
X_val_scaled = joblib.load('processed_data/X_val_scaled.pkl')
X_test_scaled = joblib.load('processed_data/X_test_scaled.pkl')

y_train = joblib.load('processed_data/y_train.pkl')
y_val = joblib.load('processed_data/y_val.pkl')
y_test = joblib.load('processed_data/y_test.pkl')

# **[4]** Model Selection Training
For the third model, we are implementing **Gradient Boosting Classifier** as an ensemble model. Unlike all other classical models implemented, we did not use Sklearn as our library. Instead, we used xgboost because compared to other libraries, this is optimized for efficiency in training models and high accuracy. Built-in factors include regularization to prevent overfitting, handling of missing values, and parallel computing capabilities. 

In theory, gradient boosting utilizes decision trees that is learned in linear sequence. It builds new trees based from learning from errors from previous decision tree to adjust hyperparameters. Compared to random forest, it is more prone to overfitting especially when regularization is not properly used, longer training time, and can consume more memory. Despite its disadvantes, it allows hhigher predictive accuracy.

**Credits**: [XGBoost Documentation](https://xgboost.readthedocs.io/en/release_3.2.0/python/python_intro.html), [XGBoost Guide to Hyperparameter Tuning](https://medium.com/@dicee/optimizing-xgboost-a-guide-to-hyperparameter-tuning-77b6e48e289d), [GeeksforGeerks: Gradient Boosting vs Random Foorest](https://www.geeksforgeeks.org/machine-learning/gradient-boosting-vs-random-forest/)

In [185]:
import sys
!{sys.executable} -m pip install xgboost

import xgboost as xgb
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats

print(f'Working on XGBoost Version: {xgb.__version__}')

Defaulting to user installation because normal site-packages is not writeable
Working on XGBoost Version: 3.2.0


In [186]:
def assess_metrics(y_true, y_pred, mode):
    accuracy = accuracy_score(y_true, y_pred)
    print(f'Accuracy in {mode} set = {round(accuracy * 100, 4)} %')
    print(f'Classification report')
    print(f'{classification_report(y_true, y_pred)}')

### Training Phase
Following baseline algorithm, after loading .pkl files created from `Preliminaries` notebook, we can create `XGBCLassifier` object based from xgboost library and then train on training set. The following are parameters of `XGBClassifier`:
1. `n_estimator`: number of boosting rounds (trees)
2. `learning_rate`: step size shrinkage to prevent overfitting
3. `max_depth`: maximum tree depth
4. `subsample`: fraction of data used for training each tree
5. `colsample_bytree`: feature subsampling ratio per tree
6. `gamma`: minimum loss function required to make a further partition on a leaf node
7. `reg_alpha`: L1 regularization, encouraging sparsity
8. `reg_lambda`: L2 regularization, encouraging reduction of weight size
9. `objective`: specifies learning task and the corresponding loss function
10. `scale_pos_weight`: adjusts weights for imbalanced datasets 

Applying from conceptual understanding of decision trees as prerequisite to understanding gradient boosting, the mentioned above (except `objective`) are hyperparameters where their values are need to be tuned during Tuning phase. 

**Credits**: [GeeksforGeeks: Gradient Boosting vs Random Forest](https://www.geeksforgeeks.org/machine-learning/gradient-boosting-vs-random-forest/), [IBM Developer: Implementing XGBoost on Python](https://developer.ibm.com/tutorials/awb-implement-xgboost-in-python/)

In [187]:
xgb_model = xgb.XGBClassifier()
xgb_model.fit(X_train_scaled, y_train)
y_pred_pretrain = xgb_model.predict(X_train_scaled)

assess_metrics(y_train, y_pred_pretrain, 'train')

Accuracy in train set = 92.9087 %
Classification report
              precision    recall  f1-score   support

           0       0.92      0.96      0.94     17358
           1       0.94      0.87      0.90     10761

    accuracy                           0.93     28119
   macro avg       0.93      0.92      0.92     28119
weighted avg       0.93      0.93      0.93     28119



### Testing if trained model is an overfit
After training the model, we can test if this works well with validation set. If a trained model performs well on training set but not on validation set, then it is most likely to be an overfit.

In [188]:
y_pred_preval = xgb_model.predict(X_val_scaled)
assess_metrics(y_val, y_pred, 'val')

Accuracy in val set = 55.3767 %
Classification report
              precision    recall  f1-score   support

           0       0.63      0.69      0.65      3720
           1       0.40      0.34      0.37      2306

    accuracy                           0.55      6026
   macro avg       0.51      0.51      0.51      6026
weighted avg       0.54      0.55      0.55      6026



In [189]:
# code to compare performances between training set and validation set and then tell if this is an overfit

# **[5]** Hyperparameter Tuning
Aimed in improving the performance of a model, this phase uses data from validation set to find the best set of possible hyperparameters in optimizing the model's performance. This method uses random search algorithm using `RandomSearchCV` library wherein each hyperparamaters are arbitrarily picked from random set of possible values using `stats` package. The best set of hyperparameters are used in creating another model.

Credits: [Medium: XGBoost Guide to Hyperparameter Tuning](https://medium.com/@dicee/optimizing-xgboost-a-guide-to-hyperparameter-tuning-77b6e48e289d), [GeeksForGeeks: XGBClassifier in Machine Learning](https://www.geeksforgeeks.org/machine-learning/xgbclassifier/)

In [190]:
param_dist = {
    'n_estimators': stats.randint(20, 100),
    'learning_rate': stats.uniform(0.01, 0.1),
    'max_depth': stats.randint(3, 10),
    'subsample': stats.uniform(0.5, 0.5),
    'colsample_bytree': stats.uniform(0.5, 0.5),
    'gamma': stats.uniform(0.4, 10),
    'reg_alpha': [0, 1],
    'reg_lambda': [0, 1],
    'scale_pos_weight': stats.uniform(0.5, 0.5)
}


# Defining randoms and training it
random_search = RandomizedSearchCV(
    estimator=xgb_model, 
    param_distributions=param_dist, 
    n_iter=10,
    scoring='f1', 
    n_jobs=-1,
    refit=False,
    cv=5,  
    random_state=42)
random_search.fit(X_val_scaled, y_val)

# Displaying best set of hyperparameters and corresponding score
best_params = random_search.best_params_
print(f'[Best set of hyperparameters]')
for key, val in best_params.items():
    print(f'  {key}: {val}')

best_xgb_model = xgb.XGBClassifier(**best_params)

[Best set of hyperparameters]
  colsample_bytree: 0.7604171300129119
  gamma: 10.01172024349349
  learning_rate: 0.09445338486781514
  max_depth: 7
  n_estimators: 72
  reg_alpha: 1
  reg_lambda: 1
  scale_pos_weight: 0.982627653632069
  subsample: 0.8035171238433423


In [191]:
# Code to compare how values of hyperparameter changed from pre-tuned model to post-tuned model

## Comparison of performances between pre-tuned and post-tuned model

In [192]:
best_xgb_model.fit(X_train_scaled, y_train)
y_pred_postval = xgb_model.predict(X_val_scaled)
assess_metrics(y_val, y_pred_postval, 'val')

Accuracy in val set = 81.8287 %
Classification report
              precision    recall  f1-score   support

           0       0.83      0.88      0.86      3720
           1       0.79      0.72      0.75      2306

    accuracy                           0.82      6026
   macro avg       0.81      0.80      0.80      6026
weighted avg       0.82      0.82      0.82      6026



In [193]:
# Code to compare performances between pre-tuned and post-tuned model

# **[6]** Model Evaluation
The testing phase serves as final performance of using tuned model to predict values.

In [194]:
y_pred_posttest = best_xgb_model.predict(X_test_scaled)
assess_metrics(y_test, y_pred_posttest, 'test')

Accuracy in test set = 81.5632 %
Classification report
              precision    recall  f1-score   support

           0       0.82      0.90      0.86      3720
           1       0.81      0.68      0.74      2306

    accuracy                           0.82      6026
   macro avg       0.81      0.79      0.80      6026
weighted avg       0.81      0.82      0.81      6026

